# Free-field 2–6× downsampling

Five time windows are selected with NumPy seed 42. All three diffusion samplers use 100 reverse steps and seed 123. Each FNO/LNO model is matched to its spatial downsampling factor.

Run cells in order. Sampling counts below are reverse-diffusion steps; checkpoint numbers identify training state. Data and checkpoints are installed under the repository root.


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'models').is_dir() and (p / 'sampler').is_dir() and (p / 'trainer').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the MRSM repository.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
OUTPUT = ROOT / 'outputs' / 'free_field_downsampling'
OUTPUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.family': 'Arial', 'font.size': 10})

def format_current_figure():
    def tick(value, pos):
        return '0' if abs(value) < 1e-12 else f'{value:g}'
    for ax in plt.gcf().axes:
        if ax.get_xscale() == 'linear':
            ax.xaxis.set_major_formatter(FuncFormatter(tick))
        if ax.get_yscale() == 'linear':
            ax.yaxis.set_major_formatter(FuncFormatter(tick))

print('Repository:', ROOT)
print('Device:', device)


In [ ]:
import os
from sampler.utils import *
import importlib
import sampler.utils
from trainer.utils import restore_checkpoint
import torch
import torch.nn as nn
import json
from collections import namedtuple
from tqdm import tqdm
from models import get_model, get_ema
from sampler import VESDE
import numpy as np
from einops import rearrange
data_name = '2DSound'
version = 'v22'
opt_path = './results/' + data_name + '_' + version + '/config.json'
with open(opt_path, 'r', encoding='utf-8') as f:
    opt = json.load(f)
    opt['num_channels'] = opt['num_components'] + opt['num_conditions']
opt['results_path'] = str(Path(opt_path).parent)
opt['device'] = device
OPT_class = namedtuple('OPT_class', opt.keys())
opt = OPT_class(**opt)
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'


In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
net = get_model(opt)
net.to(device)
ema = get_ema(net.parameters(), decay=opt.ema_rate)
optimizer = torch.optim.Adam(net.parameters(), lr=opt.lr, betas=(0.9, 0.999), eps=1e-08, weight_decay=0)
state = dict(optimizer=optimizer, model=net, ema=ema, epoch=0, iteration=0, loss_train=[])
ckpt_path = opt.results_path + '/checkpoint_200.pth'
if not Path(ckpt_path).is_file():
    raise FileNotFoundError(ckpt_path)
state = restore_checkpoint(ckpt_path, state, device)
net = state['model']
ema = state['ema']
ema.copy_to(net.parameters())
print('Model loaded from epoch', state['epoch'])


In [ ]:
class DataScalar:

    def __init__(self, mean, std, device='cpu'):
        self.mean_np = mean
        self.std_np = std
        self.device = device
        self.mean_torch = torch.tensor(mean).float().to(device)
        self.std_torch = torch.tensor(std).float().to(device)
        if self.mean_torch.ndim == 5:
            self.mean_torch = self.mean_torch.permute(0, 1, 4, 2, 3)
            self.std_torch = self.std_torch.permute(0, 1, 4, 2, 3)

    def normalize(self, x_np):
        mean_t = self.mean_np.transpose(0, 1, 4, 2, 3)
        std_t = self.std_np.transpose(0, 1, 4, 2, 3)
        return (x_np - mean_t) / std_t

    def unnormalize(self, x):
        if x.device != self.mean_torch.device:
            self.mean_torch = self.mean_torch.to(x.device)
            self.std_torch = self.std_torch.to(x.device)
        return x * self.std_torch + self.mean_torch

def select_test_sample_by_freq(data_raw, frequency_hz=375):
    frequencies = [125, 135, 175, 225, 275, 325, 375, 415, 425]
    if frequency_hz not in frequencies:
        raise ValueError(f"Choose a frequency from {frequencies}")
    index = frequencies.index(frequency_hz)
    print(f"Test frequency: {frequency_hz} Hz")
    return data_raw[index:index + 1]


In [ ]:
import sys
import os
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from models.fno import FNO2d
from models.lno import LNO2d

class LNOConfig:
    MODES = 16
    WIDTH = 8
    NUM_CHANNELS = 3
    SEQUENCE_LENGTH = 10
    INPUT_RES = 2
    OUTPUT_RES = 64
    DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    MODEL_PATH = 'results_LNO_2dsound/lno_2dsound_reconstruction_6x.pth'

class FNOConfig:
    MODES = 16
    WIDTH = 64
    NUM_CHANNELS = 3
    SEQUENCE_LENGTH = 10
    INPUT_RES = 2
    OUTPUT_RES = 64
    DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    MODEL_PATH = 'results_FNO_2dsound/fno_2dSound_reconstruction_6x.pth'

def get_grid(shape, device):
    (batchsize, size_x, size_y) = (shape[0], shape[1], shape[2])
    gridx = torch.tensor(np.linspace(0, 1, size_x), dtype=torch.float)
    gridx = gridx.reshape(1, size_x, 1, 1).repeat([batchsize, 1, size_y, 1])
    gridy = torch.tensor(np.linspace(0, 1, size_y), dtype=torch.float)
    gridy = gridy.reshape(1, 1, size_y, 1).repeat([batchsize, size_x, 1, 1])
    return torch.cat((gridx, gridy), dim=-1).to(device)

def load_fno_model(config, path_override=None):
    model = FNO2d(num_channels=config.NUM_CHANNELS, out_channels=config.NUM_CHANNELS * config.SEQUENCE_LENGTH, modes1=config.MODES, modes2=config.MODES, width=config.WIDTH, initial_step=config.SEQUENCE_LENGTH).to(config.DEVICE)
    path = path_override if path_override else config.MODEL_PATH
    if not os.path.exists(path):
        path = os.path.join('results', config.MODEL_PATH)
    if os.path.exists(path):
        state_dict = torch.load(path, map_location=config.DEVICE)
        model.load_state_dict(state_dict)
        print(f'FNO Model loaded from {path}')
    else:
        raise FileNotFoundError(path)
    model.eval()
    return model

def load_lno_model(config, path_override=None):
    tx = torch.linspace(0, 1, config.OUTPUT_RES).unsqueeze(0).to(config.DEVICE)
    ty = torch.linspace(0, 1, config.OUTPUT_RES).unsqueeze(0).to(config.DEVICE)
    model = LNO2d(in_channels=config.NUM_CHANNELS * config.SEQUENCE_LENGTH, out_channels=config.NUM_CHANNELS * config.SEQUENCE_LENGTH, width=config.WIDTH, modes1=config.MODES, modes2=config.MODES, tx=tx, ty=ty).to(config.DEVICE)
    path = path_override if path_override else config.MODEL_PATH
    if not os.path.exists(path):
        path = os.path.join('results', config.MODEL_PATH)
    if os.path.exists(path):
        state_dict = torch.load(path, map_location=config.DEVICE)
        model.load_state_dict(state_dict)
        print(f'LNO Model loaded from {path}')
    else:
        raise FileNotFoundError(path)
    model.eval()
    return model


In [ ]:
data_test_path = 'data/anechoic_high_prec_dataset_last700_test_2.npy'
scalar_path = opt.results_path + '/scalar.npy'
data_raw = np.load(data_test_path)
scalar_data = np.load(scalar_path, allow_pickle=True)
(mean_val, std_val) = (scalar_data[0], scalar_data[1]) if isinstance(scalar_data, np.ndarray) else (scalar_data[0], scalar_data[1])
scaler = DataScalar(mean_val, std_val, device=device)
x_test_raw = select_test_sample_by_freq(data_raw, frequency_hz=375)
x_test = x_test_raw.transpose(0, 1, 4, 2, 3)
print(f'Sample Shape (Model Input): {x_test.shape}')
(_, z) = np.meshgrid(np.linspace(0, 2 * np.pi, opt.image_size, endpoint=False), np.linspace(0, 2 * np.pi, opt.image_size, endpoint=False), indexing='ij')


In [ ]:
m = 2
T = opt.num_frames
T_prime = 100
B_batch = int(np.ceil((T_prime - m) / (T - m)))
T_prime = B_batch * (T - m) + m
n_samples = 1
seed_samples = 1
rnd = np.random.RandomState(seed_samples)
t_start = 0
samples = x_test[:, t_start:t_start + T_prime]
print(f'Target Shape: {samples.shape}')
samples_scale_all = scaler.normalize(samples)
print(samples_scale_all.shape)


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import os
import copy
SCALES = [2, 3, 4, 5, 6]
np.random.seed(42)
T_STARTS = np.random.choice(np.arange(0, 200), 5, replace=False)
print(f'Selected random t_starts: {T_STARTS}')
methods_ordered = ['LNO', 'FNO', 'SDE', 'ISS', 'PI-ISS']
results = {s: {m: [] for m in methods_ordered} for s in SCALES}
LNO_PATH_TEMPLATE = 'results_LNO_2dsound/lno_2dsound_reconstruction_{}x.pth'
FNO_PATH_TEMPLATE = 'results_FNO_2dsound/fno_2dSound_reconstruction_{}x.pth'

def get_grid(shape, device):
    (batchsize, size_x, size_y) = (shape[0], shape[1], shape[2])
    gridx = torch.tensor(np.linspace(0, 1, size_x), dtype=torch.float)
    gridx = gridx.reshape(1, size_x, 1, 1).repeat([batchsize, 1, size_y, 1])
    gridy = torch.tensor(np.linspace(0, 1, size_y), dtype=torch.float)
    gridy = gridy.reshape(1, 1, size_y, 1).repeat([batchsize, size_x, 1, 1])
    return torch.cat((gridx, gridy), dim=-1).to(device)

def calculate_fair_metric(pred_phys, gt_phys):
    if pred_phys is None:
        return 999.0
    if pred_phys.ndim == 5:
        pred_phys = pred_phys[0]
    T_len = min(len(pred_phys), len(gt_phys))
    p = pred_phys[:T_len]
    g = gt_phys[:T_len]
    try:
        err_p = cal_rmse(g[:, 0:1], p[:, 0:1], normalize=True)
        mean_p = np.mean(err_p)
        err_vx = cal_rmse(g[:, 1:2], p[:, 1:2], normalize=True)
        mean_vx = np.mean(err_vx)
        err_vy = cal_rmse(g[:, 2:3], p[:, 2:3], normalize=True)
        mean_vy = np.mean(err_vy)
    except NameError:
        raise
    return (mean_p + mean_vx + mean_vy) / 3.0

def unnorm_tensor(x_numpy):
    if x_numpy is None:
        return None
    t = torch.tensor(x_numpy).float().to(device)
    return scaler.unnormalize(t).detach().cpu().numpy()
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
for scale in SCALES:
    print(f"\n{'#' * 30}")
    print(f' Processing Downsampling Scale: {scale}x')
    print(f"{'#' * 30}")

    class LNOConfigCurrent:
        MODES = 16
        WIDTH = 8
        NUM_CHANNELS = 3
        SEQUENCE_LENGTH = 10
        INPUT_RES = scale
        OUTPUT_RES = 64
        DEVICE = device
        MODEL_PATH = LNO_PATH_TEMPLATE.format(scale)

    class FNOConfigCurrent:
        MODES = 16
        WIDTH = 64
        NUM_CHANNELS = 3
        SEQUENCE_LENGTH = 10
        INPUT_RES = scale
        OUTPUT_RES = 64
        DEVICE = device
        MODEL_PATH = FNO_PATH_TEMPLATE.format(scale)
    lno_model = None
    fno_model = None
    try:
        lno_model = load_lno_model(LNOConfigCurrent, path_override=LNOConfigCurrent.MODEL_PATH)
        lno_model = lno_model.to(device)
        lno_model.eval()
        print(f'LNO Model loaded from {LNOConfigCurrent.MODEL_PATH}')
    except Exception as e:
        raise
    try:
        fno_model = load_fno_model(FNOConfigCurrent, path_override=FNOConfigCurrent.MODEL_PATH)
        fno_model = fno_model.to(device)
        fno_model.eval()
        print(f'FNO Model loaded from {FNOConfigCurrent.MODEL_PATH}')
    except Exception as e:
        raise
    for (t_idx, t_start) in enumerate(T_STARTS):
        print(f'\n  >>> Trial {t_idx + 1}/5 (t_start={t_start}) ...')
        torch.cuda.empty_cache()
        T = getattr(opt, 'num_frames', 10)
        T_prime = 100
        m = 2
        B_batch = int(np.ceil((T_prime - m) / (T - m)))
        T_prime_calc = B_batch * (T - m) + m
        samples = x_test[:, t_start:t_start + T_prime_calc]
        samples_scale_all = scaler.normalize(samples)
        gt_phys = samples[0]
        obs_type = 'regular'
        if obs_type == 'regular':
            T_prime_y = T_prime_calc
            transform_gen = lambda x: x[:, :T_prime_y, :opt.num_components, ::scale, ::scale]
            sparsity = 1 - 1 / scale ** 2
        setup_seed(123)
        outer_loop = 100
        denoising_steps1 = outer_loop
        denoising_steps2 = outer_loop
        alpha_case = 0.4
        (beta, gamma) = (0.4, 0.5)
        cof = 5e-08
        sde = VESDE(opt, sigma_min=opt.beta_min, sigma_max=opt.beta_max, N=outer_loop)
        sde1 = VESDE(opt, sigma_min=opt.beta_min, sigma_max=opt.beta_max, N=denoising_steps1)
        sde2 = VESDE(opt, sigma_min=opt.beta_min, sigma_max=opt.beta_max, N=denoising_steps2)
        y = transform_gen(samples_scale_all)
        alpha = alpha_case / np.sqrt(1 - sparsity)
        latent_mask = torch.ones([T, 1, 1, 1]).float().to(device)
        obs_mask = torch.zeros([T, 1, 1, 1]).float().to(device)
        frame_indices = torch.arange(T).to(device)
        reap_fn = lambda x, num: torch.stack([x for _ in range(num)])
        net_fn = lambda x, t: net(x, x0=x, timesteps=t, latent_mask=reap_fn(latent_mask, len(x)), obs_mask=reap_fn(obs_mask, len(x)), frame_indices=reap_fn(frame_indices, len(x)))[0]
        try:
            x_sde = s3gm_sample_2d(opt, net_fn, sde, y, transform_gen, alpha=0.5 / np.sqrt(1 - sparsity), beta=beta, gamma=gamma, T_prime_y=T_prime_y, T_prime=T_prime_calc, overlap=m, corrector=NoneCorrector, n_steps=0, snr=0.128, device=device, dtype='float', eps=1e-12, probability_flow=False, continuous=True)
            phys_sde = unnorm_tensor(x_sde)
            results[scale]['SDE'].append(calculate_fair_metric(phys_sde, gt_phys))
            print(f"    SDE completed, error: {results[scale]['SDE'][-1]:.6f}")
        except Exception as e:
            raise
        setup_seed(123)
        try:
            x_iss = s3gm_sample_2d_ISS_2dsound(opt, net_fn, sde1, y, transform_gen, alpha=0.5 / np.sqrt(1 - sparsity), beta=beta, gamma=gamma, T_prime_y=T_prime_y, T_prime=T_prime_calc, overlap=m, corrector=NoneCorrector, n_steps=0, eta=0, reg_coef_pa=10 * T_prime_calc, reg_coef_ar=100.0, denoising_steps=denoising_steps1, snr=0.128, device=device, dtype='float', eps=1e-12, probability_flow=False, continuous=True)
            phys_iss = unnorm_tensor(x_iss)
            results[scale]['ISS'].append(calculate_fair_metric(phys_iss, gt_phys))
            print(f"    ISS completed, error: {results[scale]['ISS'][-1]:.6f}")
        except Exception as e:
            raise
        setup_seed(123)
        try:
            x_piiss = s3gm_sample_2d_ISS_2dsound_res(opt, net_fn, sde2, y, transform_gen, alpha=1.4 / np.sqrt(1 - sparsity), beta=beta, gamma=gamma, T_prime_y=T_prime_y, T_prime=T_prime_calc, overlap=m, corrector=NoneCorrector, n_steps=0, eta=0, use_residual_objective=False, reg_coef_pa=10 * T_prime_calc, reg_coef_ar=100.0, denoising_steps=denoising_steps2, snr=0.128, device=device, dtype='float', eps=1e-12, probability_flow=False, continuous=True, data_scalar=scaler.unnormalize, physics_start_step=100, reg_coef_wave=1 * cof, reg_coef_cont=100000000.0 * cof, reg_coef_euler=100000000.0 * cof, physics_c=343.0, physics_rho0=1.21, physics_dt=0.0001, physics_dx=1 / 21, physics_dy=1 / 21)
            phys_piiss = unnorm_tensor(x_piiss)
            results[scale]['PI-ISS'].append(calculate_fair_metric(phys_piiss, gt_phys))
            print(f"    PI-ISS completed, error: {results[scale]['PI-ISS'][-1]:.6f}")
        except Exception as e:
            raise
        for var_name in ['x_sde', 'x_iss', 'x_piiss', 'phys_sde', 'phys_iss', 'phys_piiss', 'y', 'sde', 'sde1', 'sde2']:
            if var_name in locals():
                del locals()[var_name]
        torch.cuda.empty_cache()
        if lno_model is not None:
            try:
                samples_norm_lno = scaler.normalize(samples[:, :, :opt.num_components])
                gt_input_lno = torch.tensor(samples_norm_lno, dtype=torch.float).to(device)
                SEQ_LEN = LNOConfigCurrent.SEQUENCE_LENGTH
                num_chunks = int(np.ceil(T_prime_calc / SEQ_LEN))
                pred_lno_list = []
                for i in range(num_chunks):
                    if i > 0:
                        for var in ['chunk_data_padded', 'x_low', 'x_in', 'x_in_lno', 'out_lno']:
                            if var in locals():
                                del locals()[var]
                    chunk_start = i * SEQ_LEN
                    actual_end = min((i + 1) * SEQ_LEN, gt_input_lno.shape[1])
                    chunk_data = gt_input_lno[:, chunk_start:actual_end, ...]
                    pad_len = 0
                    if chunk_data.shape[1] < SEQ_LEN:
                        pad_len = SEQ_LEN - chunk_data.shape[1]
                        (B, _, C, H, W) = chunk_data.shape
                        padding = torch.zeros(B, pad_len, C, H, W, device=device)
                        chunk_data_padded = torch.cat([chunk_data, padding], dim=1)
                    else:
                        chunk_data_padded = chunk_data
                    x_low = chunk_data_padded[:, :, :, ::scale, ::scale]
                    (B_sz, T_ch, C_ch, H_l, W_l) = x_low.shape
                    x_in = x_low.reshape(B_sz * T_ch, C_ch, H_l, W_l)
                    x_in = F.interpolate(x_in, size=(LNOConfigCurrent.OUTPUT_RES, LNOConfigCurrent.OUTPUT_RES), mode='bilinear', align_corners=True)
                    x_in_lno = x_in.reshape(B_sz, T_ch * C_ch, LNOConfigCurrent.OUTPUT_RES, LNOConfigCurrent.OUTPUT_RES)
                    with torch.no_grad():
                        out_lno = lno_model(x_in_lno)
                    out_lno = out_lno.reshape(B_sz, T_ch, C_ch, LNOConfigCurrent.OUTPUT_RES, LNOConfigCurrent.OUTPUT_RES)
                    if pad_len > 0:
                        out_lno = out_lno[:, :-pad_len, ...]
                    pred_lno_list.append(out_lno.detach().cpu())
                    torch.cuda.empty_cache()
                pred_seq_lno = torch.cat(pred_lno_list, dim=1).to(device)
                phys_lno = scaler.unnormalize(pred_seq_lno).detach().cpu().numpy()
                phys_lno_full = np.zeros_like(gt_phys)[None, ...]
                phys_lno_full[0, :, :opt.num_components] = phys_lno[0]
                results[scale]['LNO'].append(calculate_fair_metric(phys_lno_full, gt_phys))
                print(f"    LNO inference successful, error: {results[scale]['LNO'][-1]:.6f}")
                del pred_seq_lno, phys_lno, phys_lno_full, pred_lno_list
                torch.cuda.empty_cache()
            except Exception as e:
                raise
        else:
            results[scale]['LNO'].append(None)
        if fno_model is not None:
            try:
                samples_norm = scaler.normalize(samples[:, :, :opt.num_components])
                gt_input = torch.tensor(samples_norm, dtype=torch.float).permute(0, 1, 3, 4, 2).to(device)
                SEQ_LEN = FNOConfigCurrent.SEQUENCE_LENGTH
                num_chunks = int(np.ceil(T_prime_calc / SEQ_LEN))
                pred_seq_list = []
                for i in range(num_chunks):
                    if i > 0:
                        for var in ['chunk_data_padded', 'x_low', 'x_in', 'out']:
                            if var in locals():
                                del locals()[var]
                    chunk_start = i * SEQ_LEN
                    actual_end = min((i + 1) * SEQ_LEN, gt_input.shape[1])
                    chunk_data = gt_input[:, chunk_start:actual_end, ...]
                    pad_len = 0
                    if chunk_data.shape[1] < SEQ_LEN:
                        pad_len = SEQ_LEN - chunk_data.shape[1]
                        (B, _, H, W, C) = chunk_data.shape
                        padding = torch.zeros(B, pad_len, H, W, C, device=device)
                        chunk_data_padded = torch.cat([chunk_data, padding], dim=1)
                    else:
                        chunk_data_padded = chunk_data
                    x_low = chunk_data_padded[:, :, ::scale, ::scale, :]
                    (B_sz, T_ch, H_l, W_l, C_ch) = x_low.shape
                    x_in = x_low.reshape(B_sz * T_ch, H_l, W_l, C_ch).permute(0, 3, 1, 2)
                    x_in = F.interpolate(x_in, size=(FNOConfigCurrent.OUTPUT_RES, FNOConfigCurrent.OUTPUT_RES), mode='bilinear', align_corners=True)
                    x_in = x_in.reshape(B_sz, T_ch, C_ch, FNOConfigCurrent.OUTPUT_RES, FNOConfigCurrent.OUTPUT_RES)
                    x_in = x_in.permute(0, 3, 4, 1, 2)
                    x_in = x_in.reshape(B_sz, FNOConfigCurrent.OUTPUT_RES, FNOConfigCurrent.OUTPUT_RES, T_ch * C_ch)
                    grid = get_grid(x_in.shape, device=device)
                    with torch.no_grad():
                        out = fno_model(x_in, grid)
                    if out.dim() == 4:
                        out = out.unsqueeze(-2)
                    out = out.squeeze(-2)
                    out = out.view(B_sz, FNOConfigCurrent.OUTPUT_RES, FNOConfigCurrent.OUTPUT_RES, T_ch, C_ch)
                    out = out.permute(0, 3, 1, 2, 4)
                    if pad_len > 0:
                        out = out[:, :-pad_len, ...]
                    pred_seq_list.append(out.detach().cpu())
                    torch.cuda.empty_cache()
                pred_seq = torch.cat(pred_seq_list, dim=1).to(device)
                pred_phys = scaler.unnormalize(pred_seq.permute(0, 1, 4, 2, 3)).detach().cpu().numpy()
                phys_fno_full = np.zeros_like(gt_phys)[None, ...]
                phys_fno_full[0, :, :opt.num_components] = pred_phys[0]
                results[scale]['FNO'].append(calculate_fair_metric(phys_fno_full, gt_phys))
                print(f"    FNO inference successful, error: {results[scale]['FNO'][-1]:.6f}")
                del pred_seq, pred_phys, phys_fno_full, pred_seq_list
                torch.cuda.empty_cache()
            except Exception as e:
                raise
        else:
            results[scale]['FNO'].append(None)
        torch.cuda.empty_cache()
print('\n' + '=' * 80)
print('Final Results Summary:')
print('=' * 80)
for scale in SCALES:
    print(f'\nScale: {scale}x')
    print('-' * 40)
    for method in methods_ordered:
        errors = [e for e in results[scale][method] if e is not None and e < 999.0]
        if errors:
            avg_err = np.mean(errors)
            std_err = np.std(errors)
            min_err = np.min(errors)
            max_err = np.max(errors)
            print(f'  {method:6s}: Avg = {avg_err:.6f} ± {std_err:.6f}, Min = {min_err:.6f}, Max = {max_err:.6f} ({len(errors)} trials)')
        else:
            print(f'  {method:6s}: All trials failed')
import pandas as pd
from datetime import datetime
save_dir = 'outputs/free_field_downsampling/'
if not os.path.exists(save_dir):
    os.makedirs(save_dir, exist_ok=True)
csv_data = {'Scale': SCALES}
for method in methods_ordered:
    mean_list = []
    std_list = []
    trials_lists = {f'{method}_trial_{i + 1}': [] for i in range(len(T_STARTS))}
    for scale in SCALES:
        valid_errors = [e for e in results[scale][method] if e is not None and e < 999.0]
        if valid_errors:
            mean_list.append(np.mean(valid_errors))
            std_list.append(np.std(valid_errors))
        else:
            mean_list.append(np.nan)
            std_list.append(np.nan)
        for i in range(len(T_STARTS)):
            if i < len(results[scale][method]):
                val = results[scale][method][i]
                val = np.nan if val is None or val == 999.0 else val
                trials_lists[f'{method}_trial_{i + 1}'].append(val)
            else:
                trials_lists[f'{method}_trial_{i + 1}'].append(np.nan)
    csv_data[f'{method}_mean'] = mean_list
    csv_data[f'{method}_std'] = std_list
    for (k, v) in trials_lists.items():
        csv_data[k] = v
df_results = pd.DataFrame(csv_data)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
csv_file = os.path.join(save_dir, f'baselines_comparison_{timestamp}.csv')
df_results.to_csv(csv_file, index=False)
npy_file = os.path.join(save_dir, f'baselines_comparison_{timestamp}.npy')
np.save(npy_file, results)
print('\n' + '=' * 80)
print(f'[SUCCESS] 数据已成功保存到以下路径:')
print(f'  CSV表格: {csv_file}')
print(f'  NPY备份: {npy_file}')
print('=' * 80)


## Compare all trials


In [ ]:
(fig, ax) = plt.subplots(figsize=(7, 4))
for method in methods_ordered:
    vals = [np.asarray(results[s][method], dtype=float) for s in SCALES]
    ax.errorbar(SCALES, [np.mean(v) for v in vals], yerr=[np.std(v) for v in vals], marker='o', capsize=3, label=method)
ax.set(xlabel='Spatial downsampling factor', ylabel='Channel-averaged nRMSE')
ax.legend(ncol=3)
format_current_figure()
fig.savefig(OUTPUT / 'downsampling_nrmse.pdf', bbox_inches='tight')
format_current_figure()
plt.show()
